# Business Context

At Innovative Research Labs, a research organization focused on advancing LLM-based technologies, staying abreast of the latest technological developments is crucial for fostering innovation and progressing projects. By streamlining access to essential information, the organization can maintain its competitive edge in the rapidly evolving landscape of LLM technologies, driving future innovations and advancements within the industry.

However, researchers often face the challenge of sifting through a vast number of articles and publications to extract valuable insights for their work. The sheer volume of information can make it difficult to quickly locate specific details or fully comprehend complex concepts, such as the Transformer architecture, which is fundamental to their research efforts.

This case study demonstrates how a document question-answering system can effectively extract relevant insights from Jay Alammar's blog article, "The Illustrated Transformer." This use case highlights the significant advantages of leveraging Generative AI to enhance research capabilities and streamline knowledge retrieval processes.

#**LangChain Document Question Answering System**

In [ ]:
!pip install transformers faiss-cpu sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 65.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 93.4 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstallin

In [ ]:
!pip freeze > requirement.txt

## **The LangChain Pipeline**

The pipeline for converting raw unstructured data into a QA chain looks like the following:

- **Loading:** First we need to load our data. Unstructured data can be loaded from many sources. The LangChain integration hub contains the full set of loaders. Each loader returns data as a LangChain Document.
- **Splitting:** Text Splitters break Documents into splits of specified size
- **Storage:** Storage (ex: often a Vector Store) will house and often embed the splits
- **Retrieval:** The app retrieves splits from storage (ex: often with similar embeddings to the input question)
- **Generation:** An LLM produces an answer using a prompt that includes the question and the retrieved data
- **Conversation (Extension):** Hold a multi-turn conversation by adding Memory to your QA chain.

## **Step 1: Loading**

In [ ]:
from langchain.document_loaders import WebBaseLoader

loader = WebBaseLoader("http://jalammar.github.io/illustrated-transformer/")
data = loader.load()

## **Step 2: Splitting**









In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap = 0)
all_splits = text_splitter.split_documents(data)

## **Step 3: Storing**

In [ ]:
# Import FAISS from Langchain Vectorstore
from langchain.vectorstores import FAISS

In [ ]:
from langchain.llms import HuggingFacePipeline
from langchain.embeddings import HuggingFaceEmbeddings

In [ ]:
model_name = "sentence-transformers/all-mpnet-base-v2"
model_kwargs = {'device': 'cpu'}
encode_kwargs = {'normalize_embeddings': False}
hf = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

<ipython-input-5-cd350cb4f9c3>:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  hf = HuggingFaceEmbeddings(


In [ ]:
 # Creating a vector store
vectorstore = FAISS.from_documents(documents=all_splits, embedding=hf) ## hf are the hugging face embeddings

## **Step 4: Retrieval**



In [ ]:
question = "What are transformers?"
docs = vectorstore.similarity_search(question)
docs

[Document(id='d4adfaff-451e-45c0-817b-13eb7a8d0d82', metadata={'source': 'http://jalammar.github.io/illustrated-transformer/', 'title': 'The Illustrated Transformer – Jay Alammar – Visualizing machine learning one concept at a time.', 'description': "Discussions:\nHacker News (65 points, 4 comments), Reddit r/MachineLearning (29 points, 3 comments)\n\n\nTranslations: Arabic, Chinese (Simplified) 1, Chinese (Simplified) 2, French 1, French 2, Italian, Japanese, Korean, Persian, Russian, Spanish 1, Spanish 2, Vietnamese\n\nWatch: MIT’s Deep Learning State of the Art lecture referencing this post\n\nFeatured in courses at Stanford, Harvard, MIT, Princeton, CMU and others\n\n\n \n  \n\n  \n  Update: This post has now become a book! Check out LLM-book.com which contains (Chapter 3) an updated and expanded version of this post speaking about the latest Transformer models and how they've evolved in the seven years since the original Transformer (like Multi-Query Attention and RoPE Positional 

## **Step 5: Generation**

In [ ]:
question = "What are transformers?"

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, pipeline
from langchain.llms import HuggingFacePipeline
from langchain.chains import RetrievalQA

# Load model and tokenizer
model_name = "google/flan-t5-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Define the pipeline
hf_pipeline = pipeline("text2text-generation", model=model, tokenizer=tokenizer, max_length=512)

# Wrap the pipeline in LangChain's LLM class
llm = HuggingFacePipeline(pipeline=hf_pipeline)

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Device set to use cuda:0
<ipython-input-9-d8988d193647>:14: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=hf_pipeline)


In [ ]:
from langchain.chains import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(llm,retriever=vectorstore.as_retriever())
qa_chain({"query": question})

<ipython-input-10-b415171d3cc1>:4: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  qa_chain({"query": question})


{'query': 'What are transformers?',
 'result': 'In this post, we will attempt to oversimplify things a bit and introduce the concepts one by one to hopefully make it easier to understand to people without in-depth knowledge of the subject matter.'}

In [ ]:
question = " What is attention mechanism?"
qa_chain({"query": question})

{'query': ' What is attention mechanism?', 'result': 'multi-headed'}

## **Step 6: Chat**


## **Conversation Summary Memory**

There are different types of memory. Each has their own parameters, their own return types, and is useful in different scenarios

We will be using **Conversation Summary Memory**. This type of memory creates a summary of the conversation over time, which can be useful for condensing information from the conversation over time.

**Conversation Summary Memory** summarizes the conversation as it happens and stores the current summary in memory. This memory can then be used to inject the summary of the conversation so far into a prompt/chain. This memory is most useful for longer conversations, where keeping the past message history in the prompt verbatim would take up too many tokens.

In [ ]:
from langchain.memory import ConversationSummaryMemory

In [ ]:
memory = ConversationSummaryMemory(
    llm=llm,
    memory_key="chat_history",
    return_messages=True
)

<ipython-input-13-c568acba53d0>:1: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationSummaryMemory(


## **Conversational Retrieval Chain**

This is a type of chain for having a conversation based on retrieved documents. This chain takes in chat history (a list of messages) and new questions, and then returns an answer to that question. The algorithm for this chain consists of three parts:

1. **Use the chat history and the new question to create a “standalone question”.** This is done so that this question can be passed into the retrieval step to fetch relevant documents. If only the new question was passed in, then the relevant context may be lacking. If the whole conversation was passed into retrieval, there may be unnecessary information there that would distract from retrieval.

2. **This new standalone question is passed to the retriever**, and relevant documents are returned.

3. **The retrieved documents are passed to an LLM** along with either the new question (default behavior) or the original question and chat history to generate a final response.



In [ ]:
from langchain.chains import ConversationalRetrievalChain

retriever = vectorstore.as_retriever()
chat = ConversationalRetrievalChain.from_llm(
    llm,
    retriever=retriever,
    memory=memory,
    verbose=True
)

## **Demonstration**

In [ ]:
chat("Explain self-attention")



> Entering new StuffDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

If you’re familiar with RNNs, think of how maintaining a hidden state allows an RNN to incorporate its representation of previous words/vectors it has processed with the current one it’s processing. Self-attention is the method the Transformer uses to bake the “understanding” of other relevant words into the one we’re currently processing.

The second step in calculating self-attention is to calculate a score. Say we’re calculating the self-attention for the first word in this example, “Thinking”. We need to score each word of the input sentence against this word. The score determines how much focus to place on other parts of the input sentence as we encode a word at a certain position.

Self-Attention at a High Level
Do

{'question': 'Explain self-attention',
 'chat_history': [SystemMessage(content='', additional_kwargs={}, response_metadata={})],
 'answer': 'Explain how the Transformer uses self-attention to bake the “understanding” of other relevant words into the one we’re currently processing.'}

In [ ]:
chat("What is a gentler approach to transformers?")



> Entering new LLMChain chain...
Prompt after formatting:
Given the following conversation and a follow up question, rephrase the follow up question to be a standalone question, in its original language.

Chat History:

system: The Transformer uses self-attention to bake the “understanding” of other relevant words into the one we’re currently processing.
Follow Up Input: What is a gentler approach to transformers?
Standalone question:

> Finished chain.


> Entering new StuffDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

Go Forth And Transform
I hope you’ve found this a useful place to start to break the ice with the major concepts of the Transformer. If you want to go deeper, I’d suggest these next steps:

Alammar, J (2018). The Illustrated Transformer [Blog post]. Retrieved from https

{'question': 'What is a gentler approach to transformers?',
 'chat_history': [SystemMessage(content='The Transformer uses self-attention to bake the “understanding” of other relevant words into the one we’re currently processing.', additional_kwargs={}, response_metadata={})],
 'answer': 'In this post, we will attempt to oversimplify things a bit and introduce the concepts one by one to hopefully make it easier to understand to people without in-depth knowledge of the subject matter.'}

In [ ]:
chat("Where were transformers proposed?")



> Entering new LLMChain chain...
Prompt after formatting:
Given the following conversation and a follow up question, rephrase the follow up question to be a standalone question, in its original language.

Chat History:

system: The Transformer uses self-attention to bake the “understanding” of other relevant words into the one we’re currently processing.
Follow Up Input: Where were transformers proposed?
Standalone question:

> Finished chain.


> Entering new StuffDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

Alammar, J (2018). The Illustrated Transformer [Blog post]. Retrieved from https://jalammar.github.io/illustrated-transformer/

Note: If you translate any of the posts, let me know so I can link your translation to the original post. My email is in the about page.

Update: This p

{'question': 'Where were transformers proposed?',
 'chat_history': [SystemMessage(content='The Transformer uses self-attention to bake the “understanding” of other relevant words into the one we’re currently processing.', additional_kwargs={}, response_metadata={})],
 'answer': 'Attention is All You Need'}

In [ ]:
chat("What are the different layers in a typical Transformer model?")

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset




> Entering new LLMChain chain...
Prompt after formatting:
Given the following conversation and a follow up question, rephrase the follow up question to be a standalone question, in its original language.

Chat History:

system: The Transformer uses self-attention to bake the “understanding” of other relevant words into the one we’re currently processing.
Follow Up Input: What are the different layers in a typical Transformer model?
Standalone question:

> Finished chain.


> Entering new StuffDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

Go Forth And Transform
I hope you’ve found this a useful place to start to break the ice with the major concepts of the Transformer. If you want to go deeper, I’d suggest these next steps:

Alammar, J (2018). The Illustrated Transformer [Blog post]. Re

{'question': 'What are the different layers in a typical Transformer model?',
 'chat_history': [SystemMessage(content='The Transformer uses self-attention to bake the “understanding” of other relevant words into the one we’re currently processing.', additional_kwargs={}, response_metadata={})],
 'answer': '2 stacked encoders and decoders'}

In [ ]:
chat("If the vocabulary is 10,000 words, what would the width of the logits vector?")



> Entering new LLMChain chain...
Prompt after formatting:
Given the following conversation and a follow up question, rephrase the follow up question to be a standalone question, in its original language.

Chat History:

system: The Transformer uses self-attention to bake the “understanding” of other relevant words into the one we’re currently processing.
Follow Up Input: If the vocabulary is 10,000 words, what would the width of the logits vector?
Standalone question:

> Finished chain.


> Entering new StuffDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

Let’s assume that our model knows 10,000 unique English words (our model’s “output vocabulary”) that it’s learned from its training dataset. This would make the logits vector 10,000 cells wide – each cell corresponding to the score of a

{'question': 'If the vocabulary is 10,000 words, what would the width of the logits vector?',
 'chat_history': [SystemMessage(content='The Transformer uses self-attention to bake the “understanding” of other relevant words into the one we’re currently processing.', additional_kwargs={}, response_metadata={})],
 'answer': '10,000 cells wide'}

In [ ]:
chat("Explain the training process of a Transformer network in detail")



> Entering new LLMChain chain...
Prompt after formatting:
Given the following conversation and a follow up question, rephrase the follow up question to be a standalone question, in its original language.

Chat History:

system: The Transformer uses self-attention to bake the “understanding” of other relevant words into the one we’re currently processing.
Follow Up Input: Explain the training process of a Transformer network in detail
Standalone question:

> Finished chain.


> Entering new StuffDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

Recap Of Training
Now that we’ve covered the entire forward-pass process through a trained Transformer, it would be useful to glance at the intuition of training the model.
During training, an untrained model would go through the exact same forward p

{'question': 'Explain the training process of a Transformer network in detail',
 'chat_history': [SystemMessage(content='The Transformer uses self-attention to bake the “understanding” of other relevant words into the one we’re currently processing.', additional_kwargs={}, response_metadata={})],
 'answer': 'Forward-pass'}

#**Conclusion**

The implementation of a document question-answering system at Innovative Research Labs has demonstrated significant potential for enhancing research efficiency and productivity. By utilizing Generative AI to extract relevant insights from complex articles such as Jay Alammar's "The Illustrated Transformer," researchers can overcome the challenges posed by information overload. This technology not only accelerates the knowledge retrieval process but also allows researchers to focus on deeper analysis and innovation.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!find /content/drive -name "Innovative_Research_Lab.ipynb"

/content/drive/MyDrive/Colab Notebooks/Innovative_Research_Lab.ipynb
